# Textbook OCR Pipeline — Google Colab

교재 사진·스캔 이미지·PDF를 업로드하면 한국어+영어 OCR을 수행하고 TXT/JSON/ZIP으로 내려받습니다. 위에서부터 셀을 차례대로 실행하세요. 업로드한 파일은 현재 Colab 세션에서만 처리됩니다.

In [ ]:
# 1. Tesseract와 Python 패키지 설치 (처음 한 번만 실행)
!apt-get -qq update
!apt-get -qq install -y tesseract-ocr tesseract-ocr-eng tesseract-ocr-kor
!pip -q install "Pillow>=10" "PyMuPDF>=1.24" "numpy>=1.26"
!tesseract --list-langs

In [ ]:
# 2. 설정값 — 필요할 때 수정
from __future__ import annotations

import csv
import io
import json
import shutil
import subprocess
import tempfile
import zipfile
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path

import pymupdf as fitz
import numpy as np
from google.colab import files
from PIL import Image, ImageFilter, ImageOps

LANGUAGE = "kor+eng"   # 영문만 처리하려면 "eng"
PSM = 3                  # Tesseract 페이지 분할 모드
PDF_DPI = 300
MAX_WIDTH = 2400
DESKEW = True
SAVE_PROCESSED = True

WORK_DIR = Path("/content/textbook_ocr")
INPUT_DIR = WORK_DIR / "input"
OUTPUT_DIR = WORK_DIR / "output"
shutil.rmtree(WORK_DIR, ignore_errors=True)
INPUT_DIR.mkdir(parents=True)
OUTPUT_DIR.mkdir(parents=True)
print("설정 완료")

In [ ]:
# 3. 이미지, PDF 또는 ZIP 파일 업로드
uploaded = files.upload()
for original_name, data in uploaded.items():
    destination = INPUT_DIR / Path(original_name).name
    destination.write_bytes(data)
    if destination.suffix.lower() == ".zip":
        extract_dir = INPUT_DIR / destination.stem
        extract_dir.mkdir(exist_ok=True)
        with zipfile.ZipFile(destination) as archive:
            for member in archive.infolist():
                target = (extract_dir / member.filename).resolve()
                if extract_dir.resolve() not in target.parents and target != extract_dir.resolve():
                    raise ValueError(f"안전하지 않은 ZIP 경로: {member.filename}")
            archive.extractall(extract_dir)

SUPPORTED = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp", ".webp", ".pdf"}
input_files = sorted(path for path in INPUT_DIR.rglob("*") if path.is_file() and path.suffix.lower() in SUPPORTED)
if not input_files:
    raise ValueError("지원되는 이미지 또는 PDF가 없습니다.")
print(f"처리할 파일 {len(input_files)}개:")
for path in input_files:
    print(" -", path.relative_to(INPUT_DIR))

In [ ]:
# 4. OCR 함수 정의
def otsu_threshold(gray: Image.Image) -> int:
    histogram = np.asarray(gray.histogram(), dtype=np.float64)
    total = histogram.sum()
    indices = np.arange(256, dtype=np.float64)
    total_mean = float(np.dot(indices, histogram))
    background_weight = background_sum = 0.0
    best_variance, best_value = -1.0, 127
    for value in range(256):
        background_weight += histogram[value]
        if background_weight == 0:
            continue
        foreground_weight = total - background_weight
        if foreground_weight == 0:
            break
        background_sum += value * histogram[value]
        background_mean = background_sum / background_weight
        foreground_mean = (total_mean - background_sum) / foreground_weight
        variance = background_weight * foreground_weight * (background_mean - foreground_mean) ** 2
        if variance > best_variance:
            best_variance, best_value = variance, value
    return best_value

def projection_score(binary: Image.Image, angle: float) -> float:
    rotated = binary.rotate(angle, resample=Image.Resampling.BILINEAR, expand=False, fillcolor=255)
    rows = (255.0 - np.asarray(rotated, dtype=np.float32)).sum(axis=1)
    return float(np.square(np.diff(rows)).sum())

def estimate_skew(binary: Image.Image, max_degrees: float = 4.0, step: float = 0.5) -> float:
    preview = binary.copy()
    if np.count_nonzero(np.asarray(preview) < 128) < 10:
        return 0.0
    if preview.width > 1000:
        ratio = 1000 / preview.width
        preview = preview.resize((1000, max(1, round(preview.height * ratio))), Image.Resampling.BILINEAR)
    angles = np.arange(-max_degrees, max_degrees + step / 2, step)
    scores = [projection_score(preview, float(angle)) for angle in angles]
    return float(angles[int(np.argmax(scores))])

def preprocess(image: Image.Image) -> Image.Image:
    image = ImageOps.exif_transpose(image).convert("L")
    if image.width > MAX_WIDTH:
        ratio = MAX_WIDTH / image.width
        image = image.resize((MAX_WIDTH, max(1, round(image.height * ratio))), Image.Resampling.LANCZOS)
    image = ImageOps.autocontrast(image.filter(ImageFilter.MedianFilter(3)), cutoff=1)
    threshold = otsu_threshold(image)
    binary = image.point(lambda pixel: 255 if pixel > threshold else 0, mode="1").convert("L")
    if DESKEW:
        angle = estimate_skew(binary)
        if abs(angle) >= 0.25:
            binary = binary.rotate(angle, resample=Image.Resampling.BICUBIC, expand=True, fillcolor=255)
    return binary

def pages_from_file(path: Path):
    if path.suffix.lower() == ".pdf":
        scale = PDF_DPI / 72.0
        with fitz.open(path) as document:
            for page_number, page in enumerate(document, 1):
                pixmap = page.get_pixmap(matrix=fitz.Matrix(scale, scale), alpha=False)
                yield page_number, Image.frombytes("RGB", (pixmap.width, pixmap.height), pixmap.samples)
    else:
        with Image.open(path) as image:
            image.load()
            yield 1, image.copy()

def ocr_page(image: Image.Image):
    with tempfile.TemporaryDirectory(prefix="colab-ocr-") as temp_dir:
        image_path = Path(temp_dir) / "page.png"
        image.save(image_path)
        completed = subprocess.run(["tesseract", str(image_path), "stdout", "-l", LANGUAGE, "--psm", str(PSM), "tsv"], capture_output=True, text=True, encoding="utf-8", errors="replace")
    if completed.returncode != 0:
        raise RuntimeError(completed.stderr.strip() or "Tesseract OCR 실패")
    grouped, words = defaultdict(list), []
    for row in csv.DictReader(io.StringIO(completed.stdout), delimiter="\t"):
        text = (row.get("text") or "").strip()
        if not text:
            continue
        try:
            block, paragraph, line = int(row["block_num"]), int(row["par_num"]), int(row["line_num"])
            word = {"text": text, "confidence": float(row["conf"]), "left": int(row["left"]), "top": int(row["top"]), "width": int(row["width"]), "height": int(row["height"])}
        except (KeyError, TypeError, ValueError):
            continue
        grouped[(block, paragraph, line)].append(text)
        words.append(word)
    output_lines, previous_paragraph = [], None
    for (block, paragraph, _), tokens in grouped.items():
        current = (block, paragraph)
        if previous_paragraph is not None and current != previous_paragraph:
            output_lines.append("")
        output_lines.append(" ".join(tokens))
        previous_paragraph = current
    confidence_values = [word["confidence"] for word in words if word["confidence"] >= 0]
    mean_confidence = sum(confidence_values) / len(confidence_values) if confidence_values else None
    return "\n".join(output_lines).strip(), words, mean_confidence

print("OCR 함수 준비 완료")

In [ ]:
# 5. 전체 파일 OCR 실행
shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
pages_dir = OUTPUT_DIR / "pages"
processed_dir = OUTPUT_DIR / "processed"
pages_dir.mkdir(parents=True)
if SAVE_PROCESSED:
    processed_dir.mkdir(parents=True)

manifest_pages, combined = [], []
page_index = 0
for source in input_files:
    for source_page, image in pages_from_file(source):
        page_index += 1
        print(f"[{page_index}] {source.name} — page {source_page}")
        prepared = preprocess(image)
        text, words, mean_confidence = ocr_page(prepared)
        stem = f"page_{page_index:04d}"
        (pages_dir / f"{stem}.txt").write_text(text + "\n", encoding="utf-8")
        payload = {"index": page_index, "source": str(source.relative_to(INPUT_DIR)), "source_page": source_page, "mean_confidence": mean_confidence, "width": prepared.width, "height": prepared.height, "words": words}
        (pages_dir / f"{stem}.json").write_text(json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
        if SAVE_PROCESSED:
            prepared.save(processed_dir / f"{stem}.png")
        manifest_pages.append(payload)
        combined.append(text)

(OUTPUT_DIR / "combined.txt").write_text("\n\n\f\n\n".join(combined) + "\n", encoding="utf-8")
manifest = {"created_at": datetime.now(timezone.utc).isoformat(), "language": LANGUAGE, "psm": PSM, "pdf_dpi": PDF_DPI, "page_count": page_index, "pages": manifest_pages}
(OUTPUT_DIR / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print(f"완료: 총 {page_index}페이지, 출력 위치 {OUTPUT_DIR}")

In [ ]:
# 6. 합본 텍스트 미리보기
combined_text = (OUTPUT_DIR / "combined.txt").read_text(encoding="utf-8")
print(combined_text[:5000])

In [ ]:
# 7. 결과 전체를 ZIP으로 다운로드
archive_path = Path(shutil.make_archive("/content/textbook_ocr_results", "zip", OUTPUT_DIR))
print(f"다운로드 파일: {archive_path.name}")
files.download(str(archive_path))